In [1]:
import cv2
import os
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tqdm import tqdm
import tensorflow as tf


In [2]:
_path = 'Intel_Image'

In [3]:
_labels = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

In [4]:
IMGSIZE = (128, 128)
CNAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
X_tr, y_tr, X_ts, y_ts = [], [], [], []
for label in _labels:
    path = _path + '/seg_train/seg_train/' + label
    for f in sorted([_ for _ in os.listdir(path) if _.lower().endswith('.jpg')]):
        X_tr += [cv2.resize(cv2.imread(os.path.join(path,f)), IMGSIZE)]
        y_tr += [CNAMES.index(label)]

In [5]:
for label in _labels:
    path = _path + '/seg_test/seg_test/' + label
    #print(path)
    for f in sorted([_ for _ in os.listdir(path) if _.lower().endswith('.jpg')]):
        X_ts += [cv2.resize(cv2.imread(os.path.join(path,f)), IMGSIZE)]
        y_ts += [CNAMES.index(label)]

In [6]:
X_tr = np.array(X_tr)
y_tr = np.array(y_tr)
X_ts = np.array(X_ts)
y_ts = np.array(y_ts)

In [7]:
X_tr = X_tr/255.0
X_ts = X_ts/255.0

In [8]:
X_tr.shape

(14034, 128, 128, 3)

In [9]:
X_ts.shape

(3000, 128, 128, 3)

In [10]:
num_epochs = 10

In [11]:
tf.random.set_seed(0)

In [12]:
model1 = tf.keras.Sequential()
model1.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(4,4), activation='relu', data_format='channels_last'))
model1.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))
model1.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(4,4), activation='relu'))
model1.add(tf.keras.layers.MaxPool2D(pool_size=(2, 2)))
model1.add(tf.keras.layers.Flatten())
model1.add(tf.keras.layers.Dense(units=1024, activation='relu'))
model1.add(tf.keras.layers.Dense(units=6, activation='softmax'))
tf.keras.backend.clear_session()
model1.build(input_shape=(None, 128, 128, 3))
model1.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

In [13]:
model1.fit(X_tr, y_tr, epochs=num_epochs, shuffle=True)

Epoch 1/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 57s 130ms/step - accuracy: 0.5036 - loss: 1.4413
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 54s 124ms/step - accuracy: 0.7534 - loss: 0.6680
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 54s 123ms/step - accuracy: 0.8414 - loss: 0.4511
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 51s 116ms/step - accuracy: 0.8954 - loss: 0.2889
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 51s 116ms/step - accuracy: 0.9287 - loss: 0.2134
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 49s 112ms/step - accuracy: 0.9627 - loss: 0.1182
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 72s 165ms/step - accuracy: 0.9768 - loss: 0.0840
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 59s 135ms/step - accuracy: 0.9868 - loss: 0.0491
Epoch 9/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 51s 116ms/step - accuracy: 0.9896 - loss: 0.0401
Epoch 10/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 51s 116ms/step - accuracy: 0.9870 - loss: 0.0439


In [14]:
ypred = np.argmax(model1.predict(X_ts), axis=-1)
acc = sum(ypred==y_ts) / y_ts.shape[0]
print(f'Accuracy is {acc}')

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step
Accuracy is 0.74


# With Regularization and Dropout

In [15]:
reg = tf.keras.regularizers.l2(0.01)
model2 = tf.keras.Sequential()
model2.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(4,4), activation='relu', data_format='channels_last', kernel_regularizer=reg))
model2.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))
model2.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(4,4), activation='relu', kernel_regularizer=reg))
model2.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))
model2.add(tf.keras.layers.Dropout(0.2))
model2.add(tf.keras.layers.Flatten())
model2.add(tf.keras.layers.Dense(units=1024, activation='relu'))
model2.add(tf.keras.layers.Dense(units=6, activation='softmax'))
tf.keras.backend.clear_session()
model2.build(input_shape=(None, 128, 128, 3))
model2.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

In [16]:
model2.fit(X_tr, y_tr, epochs=7, shuffle=True)

Epoch 1/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 61s 136ms/step - accuracy: 0.5215 - loss: 1.5547
Epoch 2/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 58s 133ms/step - accuracy: 0.7602 - loss: 0.7647 
Epoch 3/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 60s 137ms/step - accuracy: 0.8242 - loss: 0.5776 
Epoch 4/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 51s 117ms/step - accuracy: 0.8886 - loss: 0.4184
Epoch 5/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 54s 123ms/step - accuracy: 0.9236 - loss: 0.3136
Epoch 6/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 53s 120ms/step - accuracy: 0.9412 - loss: 0.2728
Epoch 7/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 57s 130ms/step - accuracy: 0.9520 - loss: 0.2377


In [17]:
ypred = np.argmax(model2.predict(X_ts),axis=-1)
acc = sum(ypred==y_ts) / y_ts.shape[0]
print(f'Accuracy is {acc}')

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step
Accuracy is 0.754


The standard deviation goes down which makes the model more consistent in its predictions making it more reliable

# With Batch Normalization

In [18]:
reg = tf.keras.regularizers.l2(0.01)
model3 = tf.keras.Sequential()
model3.add(tf.keras.layers.Conv2D(filters=16, kernel_size=(4,4), activation='relu', data_format='channels_last', kernel_regularizer=reg))
model3.add(tf.keras.layers.BatchNormalization())
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))
model3.add(tf.keras.layers.Conv2D(filters=32, kernel_size=(4,4), activation='relu', kernel_regularizer=reg))
model3.add(tf.keras.layers.BatchNormalization())
model3.add(tf.keras.layers.MaxPool2D(pool_size=(2,2)))
model3.add(tf.keras.layers.Dropout(0.2))
model3.add(tf.keras.layers.Flatten())
model3.add(tf.keras.layers.Dense(units=1024, activation='relu'))
model3.add(tf.keras.layers.BatchNormalization())
model3.add(tf.keras.layers.Dense(units=6, activation='softmax'))
tf.keras.backend.clear_session()
model3.build(input_shape=(None, 128, 128, 3))
model3.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

In [19]:
model3.fit(X_tr, y_tr, epochs=7, shuffle=True)

Epoch 1/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 70s 153ms/step - accuracy: 0.6048 - loss: 1.6513 
Epoch 2/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 80s 182ms/step - accuracy: 0.7764 - loss: 0.7894 
Epoch 3/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 67s 152ms/step - accuracy: 0.8484 - loss: 0.5731 
Epoch 4/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 74s 169ms/step - accuracy: 0.8994 - loss: 0.4088 
Epoch 5/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 65s 148ms/step - accuracy: 0.9316 - loss: 0.3208
Epoch 6/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 71s 162ms/step - accuracy: 0.9574 - loss: 0.2350 
Epoch 7/7
439/439 ━━━━━━━━━━━━━━━━━━━━ 68s 154ms/step - accuracy: 0.9600 - loss: 0.2287 


In [20]:
ypred = np.argmax(model3.predict(X_ts),axis=-1)
acc = sum(ypred==y_ts) / y_ts.shape[0]
print(f'Accuracy is {acc}')

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
Accuracy is 0.6846666666666666


Batch normalization speeds up the convergence so it uses less epochs to reach the same performance.